In [1]:
import copy
import fnmatch
import json
import getpass
import os
import pathlib
import datetime
                    
from dask.distributed import Client, SSHCluster
from laserfarm import Retiler, DataProcessing, GeotiffWriter, MacroPipeline
from laserfarm.remote_utils import get_wdclient, get_info_remote, list_remote

# Macro-Pipeline Natura2000 Workflow - Normalization


Choose whether you want to run all input files or run the only input files listed in `filename`.

In [ ]:
path_root = pathlib.Path('/project/lidarac/Data/Natura2000')

# path to retiled files 
path_input = path_root / 'retiled'

# path to normalized files
path_output = path_input.parent / 'normalized'

run = 'from_file' # 'all', 'from_file'
filename = 'Natura2000_normalize_failed.json'  # if run is 'from_file', set name of file with input file names
assert run in ['all', 'from_file']

In [ ]:
tiles = [el for el in path_input.iterdir() if el.match('tile_*_*/')]
print('Found: {} tiles'.format(len(tiles)))
if run == 'from_file':
    with open(filename, 'r') as f:
        tiles_read = json.load(f)
    tiles_read = [path_input/f for f in tiles_read]
    # check whether all files are available on dCache
    assert all([f in tiles for f in tiles_read]), f'Some of the tiles in {filename} are not in input dir'
    tiles = tiles_read
print('Normalize: {} tiles'.format(len(tiles)))

## Setup Cluster

Setup Dask cluster used for the macro-pipeline calculation.

In [4]:
from dask.distributed import Client

client = Client("tcp://10.0.1.94:37545")
client

Connection method: Direct,
Dashboard: /proxy/8787/status,
Comm: tcp://10.0.1.94:37545,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: 1 minute ago,Total memory: 0 B


## Normalization

Generate the normalized height for each point.

In [5]:
import geopandas
gdf = geopandas.read_file("/project/lidarac/Data/Cliptest/shapefile/Clip_shape_test.shp")
gdf.explode().to_file("/project/lidarac/Data/Cliptest/shapefile/Clip_shape_exploded.shp", encoding="utf-8")


/tmp/ipykernel_1878665/392002707.py:3: FutureWarning: Currently, index_parts defaults to True, but in the future, it will default to False to be consistent with Pandas. Use `index_parts=True` to keep the current behavior and True/False to silence the warning.
  gdf.explode().to_file("/project/lidarac/Data/Cliptest/shapefile/Clip_shape_exploded.shp", encoding="utf-8")


In [71]:
# setup input dictionary to configure the normalization pipeline
normalization_input = {
    'setup_local_fs': {'input_folder': path_input.as_posix(),
                       'output_folder': path_output.as_posix()},
    'load': {'attributes': 'all'},
    # Filter out artifically high points - give overflow error when writing 
    'apply_filter': {'filter_type':'select_below',
                     'attribute': 'z',
                     'threshold': 10000.}, # remove non-physically heigh points
    # filter point cloud using polygons 
    #'apply_filter': {'filter_type':'select_polygon',
    #                 'polygon_string': '/project/lidarac/Data/Cliptest/shapefile/Clip_shape_exploded.shp',
    #                 'read_from_file': True
    #                 },                  
    'normalize': 1,
    'clear_cache' : {},
}

# write input dictionary to JSON file
with open('normalize.json', 'w') as f:
    json.dump(normalization_input, f)
    

In [ ]:
macro = MacroPipeline()

# add pipeline list to macro-pipeline object and set the corresponding labels
for tile in tiles:
    dp = DataProcessing(tile.name, label=tile.name)
    normalization_input_ = copy.deepcopy(normalization_input)
    normalization_input_['export_point_cloud'] = {'filename': '{}.laz'.format(tile.name),
                                                  'overwrite': True}
    dp.config(normalization_input_)
    macro.add_task(dp)

macro.setup_cluster(cluster="tcp://10.0.1.94:37545")

# run!
macro.run()

# save outcome results and check that no error occurred before continuing
macro.print_outcome(to_file='normalize.out')

failed = macro.get_failed_pipelines()
if failed:
    with open('Natura2000_normalize_failed.json', 'w') as f:
        json.dump([pip.label for pip in failed], f)
    raise RuntimeError('Some of the pipelines have failed')

## Terminate cluster

In [ ]:
macro.shutdown()